**Database connection test**



In [1]:
!pip install psycopg2-binary python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 134.6 MB/s eta 0:00:00


In [2]:
import os

os.environ["DB_HOST"] = "134.122.33.127"
os.environ["DB_PORT"] = "5432"
os.environ["DB_NAME"] = "amazon_products"
os.environ["DB_USER"] = "amazon_user"
os.environ["DB_PASSWORD"] = "Mevisw7St5XExL?rlz9T"

In [3]:
import psycopg2
import os

conn = psycopg2.connect(
    host=os.environ["DB_HOST"],
    port=os.environ["DB_PORT"],
    dbname=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"]
)

cur = conn.cursor()
cur.execute("SELECT 1;")
print(cur.fetchone())

cur.close()
conn.close()

(1,)


**Database Struc**

In [4]:
conn = psycopg2.connect(
    host=os.environ["DB_HOST"],
    port=os.environ["DB_PORT"],
    dbname=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"]
)

cur = conn.cursor()

cur.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
""")

tables = cur.fetchall()
print(tables)

cur.close()
conn.close()

[('amazon_product_title_embeddings',), ('amazon_products',)]


In [5]:
import psycopg2
import os

def get_connection():
    return psycopg2.connect(
        host=os.environ["DB_HOST"],
        port=os.environ["DB_PORT"],
        dbname=os.environ["DB_NAME"],
        user=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"]
    )

conn = get_connection()
cur = conn.cursor()

for table_name in ["amazon_products", "amazon_product_title_embeddings"]:
    print(f"\n===== {table_name} columns =====")
    cur.execute("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'public'
        AND table_name = %s
        ORDER BY ordinal_position;
    """, (table_name,))

    for col in cur.fetchall():
        print(col)

cur.close()
conn.close()


===== amazon_products columns =====
('asin', 'text')
('title', 'text')
('img_url', 'text')
('product_url', 'text')
('stars', 'numeric')
('reviews', 'integer')
('price', 'numeric')
('list_price', 'numeric')
('category_id', 'integer')
('is_best_seller', 'boolean')
('bought_in_last_month', 'integer')
('main_category', 'text')
('loaded_at', 'timestamp with time zone')

===== amazon_product_title_embeddings columns =====
('asin', 'text')
('title_embedding', 'USER-DEFINED')
('model_name', 'text')
('embedding_created_at', 'timestamp with time zone')


In [6]:
conn = get_connection()
cur = conn.cursor()
# first few rows
cur.execute("""
    SELECT *
    FROM amazon_products
    LIMIT 5;
""")

rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

('B07H736ZBW', "SISSY pouch panties men's lace thong G-string bikini briefs hipster hot underwear sexy for men VC", 'https://m.media-amazon.com/images/I/61ZQr-onCqL._AC_UL320_.jpg', 'https://www.amazon.com/dp/B07H736ZBW', Decimal('4.30'), 0, Decimal('17.98'), Decimal('0.00'), 110, True, 0, 'Fashion, Shoes & Luggage', datetime.datetime(2026, 7, 3, 22, 38, 58, 315481, tzinfo=datetime.timezone.utc))
('B07VD1G78F', 'Wudhu Socks | Bamboo Breathable Waterproof Socks Suitable for Various Outdoor Activities (Unisex)', 'https://m.media-amazon.com/images/I/81ELWdB3KHL._AC_UL320_.jpg', 'https://www.amazon.com/dp/B07VD1G78F', Decimal('4.60'), 0, Decimal('29.00'), Decimal('0.00'), 110, False, 0, 'Fashion, Shoes & Luggage', datetime.datetime(2026, 7, 3, 22, 38, 58, 315481, tzinfo=datetime.timezone.utc))
('B0C1BSRK84', "Men's Golf Shorts 9'' Elastic Waist Quick Dry for Flat Front Travel Casual Shorts with 5 Pockets", 'https://m.media-amazon.com/images/I/61tfi-yO2oL._AC_UL320_.jpg', 'https://www.amazo

In [7]:
conn = get_connection()
import pandas as pd
# check main category
df_categories = pd.read_sql("""
    SELECT main_category, COUNT(*) AS count
    FROM amazon_products
    GROUP BY main_category
    ORDER BY count DESC;
""", conn)

conn.close()

display(df_categories)

/tmp/ipykernel_7658/584225691.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_categories = pd.read_sql("""


,main_category,count
0,"Fashion, Shoes & Luggage",310499
1,Industrial & Scientific,154168
2,Electronics & Computers,147267
3,Home & Kitchen,133788
4,Toys & Games,114437
5,Automotive,106634
6,"Arts, Crafts & Party Supplies",105454
7,Baby Products,73439
8,Tools & Home Improvement,72571
9,Video Games,54694


**Test SQL Query**

In [8]:
def search_products_sql(
    main_category=None,
    price_min=None,
    price_max=None,
    min_stars=None,
    min_reviews=None,
    keyword=None,
    limit=20
):
    where_clauses = []
    values = []

    if main_category:
        where_clauses.append("main_category = %s")
        values.append(main_category)

    if price_min is not None:
        where_clauses.append("price >= %s")
        values.append(price_min)

    if price_max is not None:
        where_clauses.append("price <= %s")
        values.append(price_max)

    if min_stars is not None:
        where_clauses.append("stars >= %s")
        values.append(min_stars)

    if min_reviews is not None:
        where_clauses.append("reviews >= %s")
        values.append(min_reviews)

    if keyword:
        where_clauses.append("title ILIKE %s")
        values.append(f"%{keyword}%")

    where_sql = " AND ".join(where_clauses) if where_clauses else "TRUE"

    sql = f"""
        SELECT
            asin,
            title,
            price,
            stars,
            reviews,
            main_category,
            product_url,
            img_url
        FROM amazon_products
        WHERE {where_sql}
        ORDER BY stars DESC NULLS LAST, reviews DESC NULLS LAST
        LIMIT %s;
    """

    values.append(limit)

    conn = get_connection()
    df = pd.read_sql(sql, conn, params=values)
    conn.close()

    return df

In [9]:
df = search_products_sql(
    main_category="Electronics & Computers",
    price_max=50,
    min_stars=4.0,
    keyword="mouse",
    limit=10
)

display(df)

/tmp/ipykernel_7658/4132599208.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=values)


,asin,title,price,stars,reviews,main_category,product_url,img_url
0,B0C694LS5W,"JieyueJewelry Cloud Keyboard Wrist Rest Set, E...",22.99,5.0,10,Electronics & Computers,https://www.amazon.com/dp/B0C694LS5W,https://m.media-amazon.com/images/I/71SYWudU75...
1,B0C84XVV5N,"USB extension cable, (2 PACK 1FT) USB 3.0 male...",7.99,5.0,7,Electronics & Computers,https://www.amazon.com/dp/B0C84XVV5N,https://m.media-amazon.com/images/I/610tvrq6Vs...
2,B0CGZMXS4Q,Halloween Watch Band for Woman Man Compatible ...,15.99,5.0,1,Electronics & Computers,https://www.amazon.com/dp/B0CGZMXS4Q,https://m.media-amazon.com/images/I/71dFI5Ep3r...
3,B0CFF632LG,[Apple MFi Certified] Lightning to USB Camera ...,9.99,5.0,1,Electronics & Computers,https://www.amazon.com/dp/B0CFF632LG,https://m.media-amazon.com/images/I/51nnxo44qc...
4,B0BTDHH656,3.5mm Headphone to Lightning Jack USB Charger ...,12.55,5.0,0,Electronics & Computers,https://www.amazon.com/dp/B0BTDHH656,https://m.media-amazon.com/images/I/51s+x7lcie...
5,B0BYD4MRTF,"USB Switch 3.0 KVM Switch,Bi-Directional USB S...",9.09,5.0,0,Electronics & Computers,https://www.amazon.com/dp/B0BYD4MRTF,https://m.media-amazon.com/images/I/61xVQyxbeZ...
6,B0C2PNDDFS,"USB 3.0 Switch, ABLEWE USB Switch Selector 2 C...",26.99,5.0,0,Electronics & Computers,https://www.amazon.com/dp/B0C2PNDDFS,https://m.media-amazon.com/images/I/71hpROmGY5...
7,B002NUBU90,Monoprice KMV Cable - 6 Feet - Black | Molded ...,11.47,5.0,0,Electronics & Computers,https://www.amazon.com/dp/B002NUBU90,https://m.media-amazon.com/images/I/6117LmfAat...
8,B0CC2KRN5C,"Wired Keyboard and Mouse Combo for Windows, Fu...",15.88,5.0,0,Electronics & Computers,https://www.amazon.com/dp/B0CC2KRN5C,https://m.media-amazon.com/images/I/61ryrg5bFh...
9,B0CG4XPNH8,BenQ Zowie G-SR II Gaming Mouse Pad for Esports,39.99,5.0,0,Electronics & Computers,https://www.amazon.com/dp/B0CG4XPNH8,https://m.media-amazon.com/images/I/31q-f7S++A...


**Without using gpt api, manually write a json to check database query**

In [10]:
params = {
    "product_description": "gaming mouse",
    "price_min": None,
    "price_max": 50,
    "min_stars": 4.0,
    "min_reviews": None,
    "brand": None,
    "sort_by": "rating",
    "main_category": "Electronics & Computers"
}

df = search_products_sql(
    main_category=params["main_category"],
    price_min=params["price_min"],
    price_max=params["price_max"],
    min_stars=params["min_stars"],
    min_reviews=params["min_reviews"],
    keyword=params["product_description"],
    limit=10
)

display(df)

/tmp/ipykernel_7658/4132599208.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=values)


,asin,title,price,stars,reviews,main_category,product_url,img_url
0,B0CG4XPNH8,BenQ Zowie G-SR II Gaming Mouse Pad for Esports,39.99,5.0,0,Electronics & Computers,https://www.amazon.com/dp/B0CG4XPNH8,https://m.media-amazon.com/images/I/31q-f7S++A...
1,B07QRR1JYB,USB Charging Cable Replacement for Logitech G4...,9.74,4.7,584,Electronics & Computers,https://www.amazon.com/dp/B07QRR1JYB,https://m.media-amazon.com/images/I/61AwHxGL+M...
2,B0BMQG9H7X,"iLeadon Keyboard Mouse Pad Set, Large Gaming M...",22.99,4.7,0,Electronics & Computers,https://www.amazon.com/dp/B0BMQG9H7X,https://m.media-amazon.com/images/I/61BxFZM7Rz...
3,B000UEZ36W,SteelSeries QcK Gaming Mouse Pad - Medium Clot...,8.99,4.7,0,Electronics & Computers,https://www.amazon.com/dp/B000UEZ36W,https://m.media-amazon.com/images/I/41Cp6u7FdW...
4,B0BP8H4XFV,Razer DeathAdder V2 Special Edition Gaming Mou...,39.99,4.7,0,Electronics & Computers,https://www.amazon.com/dp/B0BP8H4XFV,https://m.media-amazon.com/images/I/71zdbu+hxc...
5,B09XGZ4332,"MIBITRI Keyboard Mouse Pad Set, Extended Gamin...",19.99,4.7,0,Electronics & Computers,https://www.amazon.com/dp/B09XGZ4332,https://m.media-amazon.com/images/I/71jRtt5+6o...
6,B00WAA2704,SteelSeries QcK Gaming Mouse Pad - XXL Thick C...,23.08,4.7,0,Electronics & Computers,https://www.amazon.com/dp/B00WAA2704,https://m.media-amazon.com/images/I/5174u2mEzS...
7,B07S4CP96V,Ergonomic Mouse Pad 3 Pack with Wrist Rest Sup...,15.99,4.6,0,Electronics & Computers,https://www.amazon.com/dp/B07S4CP96V,https://m.media-amazon.com/images/I/71fMshDqqQ...
8,B07VRQWBMR,Glorious Gaming Mouse - Model O Minus 58 g Sup...,47.99,4.6,0,Electronics & Computers,https://www.amazon.com/dp/B07VRQWBMR,https://m.media-amazon.com/images/I/71iS3zaTAA...
9,B0BW997Z8F,Gimars 3 in1 Mouse Pad and Keyboard Wrist Rest...,18.97,4.5,0,Electronics & Computers,https://www.amazon.com/dp/B0BW997Z8F,https://m.media-amazon.com/images/I/71o-IfG6Ts...


# **Real pipeline**

In [11]:
!pip install -U openai

In [12]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

Paste your OpenAI API key: ··········


In [13]:
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

response = client.responses.create(
    model="gpt-5.4-nano",
    input="Say 'OpenAI API connected successfully.'"
)

print(response.output_text)

OpenAI API connected successfully.


In [14]:
import json
# query constructor
def construct_query_json(user_input):
    schema = {
        "type": "object",
        "properties": {
            "product_description": {
                "type": ["string", "null"],
                "description": "A concise product search phrase, such as 'wireless gaming mouse'."
            },
            "price_min": {"type": ["number", "null"]},
            "price_max": {"type": ["number", "null"]},
            "min_stars": {"type": ["number", "null"]},
            "min_reviews": {"type": ["integer", "null"]},
            "brand": {"type": ["string", "null"]},
            "sort_by": {
                "type": "string",
                "enum": [
                    "relevance",
                    "price_low_to_high",
                    "price_high_to_low",
                    "rating",
                    "popularity"
                ]
            }
        },
        "required": [
            "product_description",
            "price_min",
            "price_max",
            "min_stars",
            "min_reviews",
            "brand",
            "sort_by"
        ],
        "additionalProperties": False
    }

    response = client.responses.create(
        model="gpt-5.4-nano",
        input=[
            {
                "role": "system",
                "content": (
                    "You are a shopping query parser. "
                    "Extract structured product search parameters from the user's request. "
                    "Return only valid JSON. "
                    "If the user says 'cheap' but gives no exact price, set price_max to null. "
                    "If the user says 'good reviews' or 'high rated', set min_stars to 4.0. "
                    "If the user asks for popular products, set sort_by to popularity."
                )
            },
            {
                "role": "user",
                "content": user_input
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "shopping_query",
                "schema": schema,
                "strict": True
            }
        }
    )

    return json.loads(response.output_text)

In [15]:
# check if gpt extract paras
user_input = "I want a wireless gaming mouse under $50 with good reviews."

params = construct_query_json(user_input)
params

{'product_description': 'wireless gaming mouse',
 'price_min': None,
 'price_max': 50,
 'min_stars': 4,
 'min_reviews': None,
 'brand': None,
 'sort_by': 'relevance'}

In [16]:
def search_products_from_params(params, limit=20):
    where_clauses = []
    values = []

    if params.get("main_category"):
        where_clauses.append("main_category = %s")
        values.append(params["main_category"])

    if params.get("price_min") is not None:
        where_clauses.append("price >= %s")
        values.append(params["price_min"])

    if params.get("price_max") is not None:
        where_clauses.append("price <= %s")
        values.append(params["price_max"])

    if params.get("min_stars") is not None:
        where_clauses.append("stars >= %s")
        values.append(params["min_stars"])

    if params.get("min_reviews") is not None:
        where_clauses.append("reviews >= %s")
        values.append(params["min_reviews"])

    if params.get("brand"):
        where_clauses.append("title ILIKE %s")
        values.append(f"%{params['brand']}%")

    if params.get("product_description"):
        where_clauses.append("title ILIKE %s")
        values.append(f"%{params['product_description']}%")

    where_sql = " AND ".join(where_clauses) if where_clauses else "TRUE"

    sort_by = params.get("sort_by", "relevance")

    if sort_by == "price_low_to_high":
        order_sql = "price ASC NULLS LAST"
    elif sort_by == "price_high_to_low":
        order_sql = "price DESC NULLS LAST"
    elif sort_by == "rating":
        order_sql = "stars DESC NULLS LAST, reviews DESC NULLS LAST"
    elif sort_by == "popularity":
        order_sql = "reviews DESC NULLS LAST, stars DESC NULLS LAST"
    else:
        order_sql = "stars DESC NULLS LAST, reviews DESC NULLS LAST"

    sql = f"""
        SELECT
            asin,
            title,
            price,
            stars,
            reviews,
            main_category,
            product_url,
            img_url
        FROM amazon_products
        WHERE {where_sql}
        ORDER BY {order_sql}
        LIMIT %s;
    """

    values.append(limit)

    conn = get_connection()
    df = pd.read_sql(sql, conn, params=values)
    conn.close()

    return df, sql, values

In [17]:
!pip install -U transformers peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 159.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
from huggingface_hub import login
login()

In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "google/gemma-3-1b-it"

ADAPTER_PATH = "/content/drive/MyDrive/shopping_assistant_gemma_lora/outputs/gemma-3-1b-it-main-category-lora"

gemma_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

if gemma_tokenizer.pad_token is None:
    gemma_tokenizer.pad_token = gemma_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype=torch.bfloat16,
)

category_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

category_model.eval()

print("Fine-tuned Gemma LoRA model loaded.")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Fine-tuned Gemma LoRA model loaded.


In [21]:
label_list = [
    "Arts, Crafts & Party Supplies",
    "Automotive",
    "Baby Products",
    "Beauty & Personal Care",
    "Electronics & Computers",
    "Fashion, Shoes & Luggage",
    "Gift Cards",
    "Health & Household",
    "Home & Kitchen",
    "Industrial & Scientific",
    "Pet Supplies",
    "Smart Home",
    "Sports & Outdoors",
    "Tools & Home Improvement",
    "Toys & Games",
    "Video Games"
]

In [22]:
def clean_label(text):
    text = text.strip()
    text = text.split("\n")[0].strip()
    text = text.replace("Answer:", "").strip()
    text = text.strip("\"'")
    return text


def normalize_label(label):
    label = clean_label(label)

    mapping = {
        "Electronics": "Electronics & Computers",
        "Computers": "Electronics & Computers",
        "Computer": "Electronics & Computers",

        "Fashion Footwear": "Fashion, Shoes & Luggage",
        "Fashion": "Fashion, Shoes & Luggage",
        "Shoes": "Fashion, Shoes & Luggage",
        "Luggage": "Fashion, Shoes & Luggage",

        "Home": "Home & Kitchen",
        "Kitchen": "Home & Kitchen",

        "Toys": "Toys & Games",
        "Games": "Toys & Games",
    }

    return mapping.get(label, label)


def predict_main_category(user_input):
    labels_text = "\n".join([f"- {label}" for label in label_list])

    prompt = f"""You are a product classification assistant.
Classify the user's product search request into exactly one main category.
Output only one label from the allowed label list.

Allowed labels:
{labels_text}

User request:
{user_input}

Answer:
"""

    inputs = gemma_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(category_model.device)

    with torch.no_grad():
        outputs = category_model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=gemma_tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = gemma_tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    pred = normalize_label(response)

    # 如果模型输出不在 16 个合法标签里，返回 None 或者做 fallback
    if pred not in label_list:
        print("Warning: invalid category predicted:", pred)
        return None

    return pred

In [23]:
# test gemma classify without SQL
user_input = "I want a wireless gaming mouse under $50 with good reviews."

pred_category = predict_main_category(user_input)

print(pred_category)

Video Games


In [24]:
params = construct_query_json(user_input)

params["main_category"] = predict_main_category(user_input)

df, sql, values = search_products_from_params(params, limit=10)

print(params)
display(df)

/tmp/ipykernel_7658/3182495008.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=values)


{'product_description': 'wireless gaming mouse', 'price_min': None, 'price_max': 50, 'min_stars': 4, 'min_reviews': None, 'brand': None, 'sort_by': 'relevance', 'main_category': 'Video Games'}


,asin,title,price,stars,reviews,main_category,product_url,img_url
0,B0CCDFZ9HF,PHYLINA S450 Wireless Gaming Mouse Ultra Light...,49.99,5.0,0,Video Games,https://www.amazon.com/dp/B0CCDFZ9HF,https://m.media-amazon.com/images/I/71X5oPxTpb...
1,B08ZT34W4N,Replacement USB Charging Cable Compatible with...,6.99,5.0,0,Video Games,https://www.amazon.com/dp/B08ZT34W4N,https://m.media-amazon.com/images/I/61408s1Ru7...
2,B0CB3WS9ZW,Superglide2 - New Controllable Speed Surface F...,24.95,5.0,0,Video Games,https://www.amazon.com/dp/B0CB3WS9ZW,https://m.media-amazon.com/images/I/61+8c-tcxv...
3,B0C4G9HSBJ,1 Set Logitech G502 Lightspeed Wireless Gaming...,4.99,5.0,0,Video Games,https://www.amazon.com/dp/B0C4G9HSBJ,https://m.media-amazon.com/images/I/51VPiOGdsr...
4,B0CBFGBLVT,USB Braided Mouse Charging Cable Compatible wi...,15.99,5.0,0,Video Games,https://www.amazon.com/dp/B0CBFGBLVT,https://m.media-amazon.com/images/I/61FGijyPuM...
5,B0CC5B7N79,BestParts New Micro-USB to USB Extension Port ...,9.99,5.0,0,Video Games,https://www.amazon.com/dp/B0CC5B7N79,https://m.media-amazon.com/images/I/31SEZ2X60d...
6,B0C535M338,"Sleeper, Wireless Gaming Mouse,19000 DPI, 53g ...",49.99,5.0,0,Video Games,https://www.amazon.com/dp/B0C535M338,https://m.media-amazon.com/images/I/51dFgF6GDN...
7,B09ZQRV2FW,Hard Mouse Case for Razer Viper Ultimate/Razer...,9.99,5.0,0,Video Games,https://www.amazon.com/dp/B09ZQRV2FW,https://m.media-amazon.com/images/I/71SXdZnx6P...
8,B0BQDHS4GP,Replacement 10g Weight with Weight Door Cover ...,9.99,5.0,0,Video Games,https://www.amazon.com/dp/B0BQDHS4GP,https://m.media-amazon.com/images/I/71b+NFhWQN...
9,B0C3H2HF1K,Mchoi Hard Carrying Case Suitable for Logitech...,12.87,5.0,0,Video Games,https://www.amazon.com/dp/B0C3H2HF1K,https://m.media-amazon.com/images/I/81ikF75Z-d...


In [25]:
# package to a complete function
def shopping_assistant_pipeline(user_input, limit=10):
    params = construct_query_json(user_input)
    params["main_category"] = predict_main_category(user_input)

    df, sql, values = search_products_from_params(params, limit=limit)

    return {
        "user_input": user_input,
        "query_params": params,
        "sql": sql,
        "sql_values": values,
        "results": df
    }

In [26]:
# test packaged function
result = shopping_assistant_pipeline(
    "I want a wireless gaming mouse under $50 with good reviews.",
    limit=10
)

print(result["query_params"])
display(result["results"])

/tmp/ipykernel_7658/3182495008.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=values)


{'product_description': 'wireless gaming mouse', 'price_min': None, 'price_max': 50, 'min_stars': 4, 'min_reviews': None, 'brand': None, 'sort_by': 'relevance', 'main_category': 'Video Games'}


,asin,title,price,stars,reviews,main_category,product_url,img_url
0,B0CB3WS9ZW,Superglide2 - New Controllable Speed Surface F...,24.95,5.0,0,Video Games,https://www.amazon.com/dp/B0CB3WS9ZW,https://m.media-amazon.com/images/I/61+8c-tcxv...
1,B0C4G9HSBJ,1 Set Logitech G502 Lightspeed Wireless Gaming...,4.99,5.0,0,Video Games,https://www.amazon.com/dp/B0C4G9HSBJ,https://m.media-amazon.com/images/I/51VPiOGdsr...
2,B0CBFGBLVT,USB Braided Mouse Charging Cable Compatible wi...,15.99,5.0,0,Video Games,https://www.amazon.com/dp/B0CBFGBLVT,https://m.media-amazon.com/images/I/61FGijyPuM...
3,B08ZT34W4N,Replacement USB Charging Cable Compatible with...,6.99,5.0,0,Video Games,https://www.amazon.com/dp/B08ZT34W4N,https://m.media-amazon.com/images/I/61408s1Ru7...
4,B0BQDHS4GP,Replacement 10g Weight with Weight Door Cover ...,9.99,5.0,0,Video Games,https://www.amazon.com/dp/B0BQDHS4GP,https://m.media-amazon.com/images/I/71b+NFhWQN...
5,B0CCDFZ9HF,PHYLINA S450 Wireless Gaming Mouse Ultra Light...,49.99,5.0,0,Video Games,https://www.amazon.com/dp/B0CCDFZ9HF,https://m.media-amazon.com/images/I/71X5oPxTpb...
6,B09ZQRV2FW,Hard Mouse Case for Razer Viper Ultimate/Razer...,9.99,5.0,0,Video Games,https://www.amazon.com/dp/B09ZQRV2FW,https://m.media-amazon.com/images/I/71SXdZnx6P...
7,B0C535M338,"Sleeper, Wireless Gaming Mouse,19000 DPI, 53g ...",49.99,5.0,0,Video Games,https://www.amazon.com/dp/B0C535M338,https://m.media-amazon.com/images/I/51dFgF6GDN...
8,B0CC5B7N79,BestParts New Micro-USB to USB Extension Port ...,9.99,5.0,0,Video Games,https://www.amazon.com/dp/B0CC5B7N79,https://m.media-amazon.com/images/I/31SEZ2X60d...
9,B0C3H2HF1K,Mchoi Hard Carrying Case Suitable for Logitech...,12.87,5.0,0,Video Games,https://www.amazon.com/dp/B0C3H2HF1K,https://m.media-amazon.com/images/I/81ikF75Z-d...
